# Hyperparameter Tuning for K-Nearest Neighbors

In this notebook, we evaluate different values of **K** to determine the optimal number of neighbors for the KNN classifier.

The model is evaluated using:

- Accuracy
- Precision
- Recall (Sensitivity)
- F1-Score

The objective is to select the value of **K** that provides the best overall performance while maintaining a high recall for detecting heart disease.

## Step 76: Import libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

## Step 77: Load and preprocess the dataset

In [ ]:
columns = [
    "age","sex","cp","trestbps","chol","fbs",
    "restecg","thalach","exang","oldpeak",
    "slope","ca","thal","target"
]

df = pd.read_csv("../data/heart.csv", header=None, names=columns)

df.replace("?", np.nan, inplace=True)

df["ca"] = pd.to_numeric(df["ca"])
df["thal"] = pd.to_numeric(df["thal"])

df.fillna(df.median(numeric_only=True), inplace=True)

df["target"] = df["target"].apply(lambda x: 0 if x == 0 else 1)

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Step 78: Create empty lists

In [ ]:
k_values = []
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

## Step 79: Train models for K = 1 to 31

In [ ]:
for k in range(1, 32):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    
    k_values.append(k)
    accuracy_scores.append(accuracy_score(y_test, prediction))
    precision_scores.append(precision_score(y_test, prediction))
    recall_scores.append(recall_score(y_test, prediction))
    f1_scores.append(f1_score(y_test, prediction))

## Step 80: Display results in a table

In [ ]:
results = pd.DataFrame({
    "K": k_values,
    "Accuracy": accuracy_scores,
    "Precision": precision_scores,
    "Recall": recall_scores,
    "F1 Score": f1_scores
})

results

## Step 81: Find the best K

In [ ]:
best_row = results.loc[
    results["Accuracy"].idxmax()
]

best_row

## Step 82: Print the best K

In [ ]:
print(f"Best K : {best_row['K']}")
print(f"Accuracy : {best_row['Accuracy']:.2%}")
print(f"Recall : {best_row['Recall']:.2%}")

## Step 83 & 84: Plot and Save Accuracy vs K

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(
    k_values,
    accuracy_scores,
    marker="o"
)
plt.title("Accuracy vs Number of Neighbors")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.grid(True)
plt.savefig(
    "../reports/figures/accuracy_vs_k.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Step 85: Plot Recall vs K

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(
    k_values,
    recall_scores,
    marker="o"
)
plt.title("Recall vs Number of Neighbors")
plt.xlabel("K")
plt.ylabel("Recall")
plt.grid(True)
plt.show()

## Step 86: Plot F1 Score vs K

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(
    k_values,
    f1_scores,
    marker="o"
)
plt.title("F1 Score vs Number of Neighbors")
plt.xlabel("K")
plt.ylabel("F1 Score")
plt.grid(True)
plt.show()

## Step 87: Retrain using the best K

In [ ]:
best_k = int(best_row["K"])
final_model = KNeighborsClassifier(
    n_neighbors=best_k
)
final_model.fit(X_train, y_train)
final_prediction = final_model.predict(X_test)

## Step 88: Final evaluation

In [ ]:
print(classification_report(
    y_test,
    final_prediction
))

## Conclusion

The hyperparameter tuning process evaluated multiple values of **K** to determine the optimal number of neighbors.

Rather than selecting a default value, the final model was chosen based on experimental results. This approach improves reproducibility and demonstrates systematic model optimization.